In [6]:
%load_ext autoreload
%autoreload 2

import os
import numpy as np
import matplotlib.pyplot as plt
from pycocotools.coco import COCO
import statsmodels.api as sm
import scipy.stats as stats
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

import pandas as pd
from collections import Counter
import numpy as np

import matplotlib.pyplot as plt

from compcats.datasets.fmri import nsd
from compcats.utils import model_utils, seed_utils
from compcats.datasets.fmri.utils import nsd_ds_utils

import main_analysis as ma
import paths

# Set seed
SEED = 42
seed_utils.seed_everything(SEED)

# Directories
RESULTS_DIR = 'results/interpreting/outliers'
os.makedirs(RESULTS_DIR, exist_ok=True)
MODELS_DIR = 'models'
os.makedirs(MODELS_DIR, exist_ok=True)
# Paths
log_file_path = os.path.join(RESULTS_DIR, 'nsd_outliers.log')
nsd_results_df_name = 'nsd_outliers.csv'

# Variables
batch_size = 2048
frois = ['FFA', 'PPA', 'EBA']
subjects = [1, 2, 5, 7]
localizer = 'grillspector'
tthreshold = 7
model_name = 'resnet50imagenet1kv2'

# COCO
coco_animal_labels = ['bird', 'cat', 'dog', 'horse', 'sheep', 'cow', 'elephant', 'bear', 'zebra', 'giraffe']
coco_face_labels = coco_animal_labels + ['person']
coco_train = COCO(paths.COCO_TRAIN_ANNOTATIONS_PATH)
coco_val = COCO(paths.COCO_VAL_ANNOTATIONS_PATH)

# NSD IDs
stim1000_nsdids = nsd_ds_utils.get_subset_nsdids(paths.NSD_DIR, 1000)

# Outliers
# Parameters
standardized_residual_threshold = 2.58

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload
loading annotations into memory...
Done (t=10.63s)
creating index...
index created!
loading annotations into memory...
Done (t=0.34s)
creating index...
index created!


In [2]:
# Util functions
def get_image_labels(image_id, coco_obj):
    """
    Get the labels for an image given its ID from the COCO dataset.
    
    Args:
        image_id (int): The ID of the image.
        
    Returns:
        list: A list of labels (category names) associated with the image.
    """
    # Get all annotations for the given image
    annotation_ids = coco_obj.getAnnIds(imgIds=image_id)
    annotations = coco_obj.loadAnns(annotation_ids)
    
    # Get category IDs from annotations
    category_ids = {annotation['category_id'] for annotation in annotations}
    
    # Get category names from category IDs
    category_names = [coco_obj.loadCats(cat_id)[0]['name'] for cat_id in category_ids]
    
    return category_names

def add_image_labels_to_df(df):
    """
    Add labels to the DataFrame given the COCO object.
    
    Args:
        df (pd.DataFrame): The DataFrame to which to add the labels.
        
    Returns:
        pd.DataFrame: The DataFrame with the labels added.
    """
    # Create a copy of the dataframe to avoid the SettingWithCopyWarning
    df_copy = df.copy()
    
    # Create a function to get the appropriate COCO object based on the split
    def get_coco_obj(split):
        if split == 'train2017':
            return coco_train
        else:
            return coco_val
        
    # Apply the function to get labels for each row using the appropriate COCO object
    df_copy.loc[:, 'labels'] = df_copy.apply(
        lambda row: get_image_labels(row['cocoId'], get_coco_obj(row['cocoSplit'])), 
        axis=1
    )
    return df_copy

def min_max_normalize(array):
    """
    Normalize the input array using min-max normalization.

    Parameters:
    array (numpy.ndarray or torch.Tensor): The input array to normalize.

    Returns:
    numpy.ndarray or torch.Tensor: The normalized array.
    """
    min_val = array.min()
    max_val = array.max()
    normalized_array = (array - min_val) / (max_val - min_val)
    return normalized_array

def z_score_normalize(array):
    """
    Normalize the input array using z-score normalization.

    Parameters:
    array (numpy.ndarray or torch.Tensor): The input array to normalize.

    Returns:
    numpy.ndarray or torch.Tensor: The normalized array.
    """
    mean_val = array.mean()
    std_val = array.std()
    normalized_array = (array - mean_val) / std_val
    return normalized_array

# NSD: Subjectwise 9000 stimuli

In [7]:
for froi in frois:
    froi_results_df_name = nsd_results_df_name.replace('.csv', f'_{froi}.csv')
    # Check if the results file already exists
    if os.path.exists(os.path.join(RESULTS_DIR, froi_results_df_name)):
        print(f"Results file {froi_results_df_name} already exists. Skipping...")
        continue
    
    # Initialize the results dataframe
    results_df = None

    model = ma.get_model(model_name)
    all_models_layer = ma.get_froi_models_layers(froi, froi)
    layer = all_models_layer[model_name]

    model_allnsdstim_activations = model_utils.get_dataset_activations(
        MODELS_DIR, paths.NSD_DIR, model, layer, dataset='nsd', froi=froi, tthreshold=tthreshold, \
        froi_localizer=localizer, froi_localizer_dir=ma.get_localizer_dir(localizer), \
        batch_size=batch_size, device='cpu'
    )

    for subject in subjects:
        #
        # Subject response
        #
        # Get subject data
        subject_data, subject_stim_map = nsd.get_betas(
            paths.NSD_DIR, subject, froi=froi, tthreshold=tthreshold, voxels=False, repetition_mean=True
        )
        # Filter out nsdIds that are in stim1000_nsdids
        subject_stim_map_9000 = subject_stim_map[~subject_stim_map['nsdId'].isin(stim1000_nsdids)]
        subject_data_9000 = subject_data[~subject_stim_map['nsdId'].isin(stim1000_nsdids)]
        # Add labels
        subject_stim_map_9000_labels = add_image_labels_to_df(subject_stim_map_9000)

        #
        # Model response
        #
        # Get model data
        model_subject_data_9000 = model_allnsdstim_activations.mean(axis=-1)[list(subject_stim_map_9000['nsdId'])]

        #
        # OLS
        #
        # Normalize the data
        normalized_model_data = z_score_normalize(model_subject_data_9000)
        normalized_subject_data = z_score_normalize(subject_data_9000)
        X = normalized_model_data.numpy()
        y = normalized_subject_data.numpy()
        # Outlier detection
        n = len(X)  # number of data points
        k = 1   # number of predictors
        # Degrees of freedom
        df = n - k - 1
        # Two-tailed p-value
        p_value = 2 * (1 - stats.t.cdf(standardized_residual_threshold, df))
        # print(f"P-value: {p_value:.4f}")
        # Fit OLS
        # Fit the model using statsmodels (now with intercept)
        X_with_intercept = sm.add_constant(X)
        ols_model = sm.OLS(y, X_with_intercept).fit()
        slope = ols_model.params[1].item()
        intercept = ols_model.params[0].item()
        # Add OLS summary to log file
        with open(log_file_path, 'a') as log_file:
            log_file.write(f"OLS summary for subject {subject}, froi {froi}:\n")
            log_file.write(ols_model.summary().as_text())
            log_file.write("\n\n")
        # print(ols_model.summary())
        standardized_residuals = ols_model.get_influence().resid_studentized_internal

        # Outlier parameters
        standardized_residual_threshold = 2.58
        n = len(X)  # number of data points
        k = 1   # number of predictors
        # Degrees of freedom
        df = n - k - 1
        # Two-tailed p-value
        p_value = 2 * (1 - stats.t.cdf(standardized_residual_threshold, df))
        # print(f"P-value: {p_value:.4f}")

        #
        # Save the results
        #
        # Create the subject results dataframe
        subject_results_df = subject_stim_map_9000_labels.copy()
        # Drop the Unnamed: 0 column which is just an index
        subject_results_df = subject_results_df.drop(columns=['Unnamed: 0'])
        # Add froi
        subject_results_df['froi'] = froi
        # Add threshold
        subject_results_df['tthreshold'] = tthreshold
        # Add localizer
        subject_results_df['model_localizer'] = localizer
        # Add subject number
        subject_results_df['subject'] = subject
        # Add model name
        subject_results_df['model_name'] = model_name
        # Add model layer
        subject_results_df['model_layer'] = layer
        # Add the model and subject data
        subject_results_df['model_response'] = model_subject_data_9000
        subject_results_df['subject_response'] = subject_data_9000
        # Add the normalized data
        subject_results_df['model_response_zscored'] = normalized_model_data
        subject_results_df['subject_response_zscored'] = normalized_subject_data
        # Add OLS details
        subject_results_df['ols_slope'] = slope
        subject_results_df['ols_intercept'] = intercept
        subject_results_df['ols_residuals'] = ols_model.resid
        subject_results_df['ols_standardized_residuals'] = standardized_residuals
        # Add predicted values from the OLS model
        subject_results_df['ols_predicted_values'] = ols_model.predict()
        # Add the R-squared value
        subject_results_df['ols_r_squared'] = ols_model.rsquared
        # Outliers
        # Add standardized residual outlier threshold
        subject_results_df['ols_standardized_residuals_threshold_tvalue'] = standardized_residual_threshold
        # Add the p-value
        subject_results_df['ols_standardized_residuals_threshold_pvalue'] = p_value
        # Add boolean flag for outliers (standardized residuals above or below standardized_residual_threshold)
        subject_results_df['is_outlier'] = abs(standardized_residuals) > standardized_residual_threshold
        # Add if the outlier is positive or negative
        subject_results_df['outlier_subject>model'] = standardized_residuals > standardized_residual_threshold
        subject_results_df['outlier_model>subject'] = standardized_residuals < -standardized_residual_threshold

        # Join the subject results to overall results df
        if results_df is None:
            results_df = subject_results_df
        else:
            # Concatenate the dataframes
            results_df = pd.concat([results_df, subject_results_df], ignore_index=True)
    # Save the results
    results_df.to_csv(
        os.path.join(RESULTS_DIR, froi_results_df_name),
        index=False
    )

Results file nsd_outliers_FFA.csv already exists. Skipping...
Results file nsd_outliers_PPA.csv already exists. Skipping...
Results file nsd_outliers_EBA.csv already exists. Skipping...


In [4]:
# All Stimuli
allstim, _ = nsd.get_subset_stimuli(paths.NSD_DIR)